# Asteroid Pole Prediction with DeLPHI

This notebook demonstrates how to use **DeLPHI** (`lc_pipeline`) to predict asteroid rotation poles from lightcurve data.

## What This Notebook Does

1. **Install** DeLPHI and dependencies
2. **Load** asteroid lightcurve data (DAMIT format or custom)
3. **Estimate** rotation period using Lomb-Scargle + Bayesian consensus
4. **Predict** pole orientation using deep learning (K=3 candidates)
5. **Visualize** results

## Expected Performance

- **Oracle@K=3 Mean**: 19.02 deg (best of 3 poles, asteroid-level, 5-fold CV on 174 DAMIT asteroids)
- **Oracle@K=3 Pooled Median**: 16.61 deg
- **ZTF External**: 18.82 deg on 163 asteroids

**Note**: The model outputs 9 unranked candidates (3 period aliases x 3 poles). There is no quality head. Users should evaluate all candidates.

---

## Step 1: Installation

Install lc_pipeline from GitHub (or use local installation).

In [ ]:
# Install from GitHub (uncomment if needed)
# !pip install -q git+https://github.com/hangbin9/DeLPHI_lc.git

# Or install from local directory
# !pip install -q -e /path/to/DeLPHI_lc

# For this demo, we assume lc_pipeline is already installed
import sys
print(f"Python: {sys.version}")

# Import required libraries
import numpy as np
import pandas as pd
import json
from pathlib import Path

try:
    from lc_pipeline import analyze
    print("lc_pipeline imported successfully")
except ImportError as e:
    print(f"Failed to import lc_pipeline: {e}")
    print("\nPlease install first:")
    print("  !pip install -q git+https://github.com/hangbin9/DeLPHI_lc.git")

## Step 2: Load Example Data

We'll use asteroid 101 Adrastea from the DAMIT database.

In [ ]:
# Option 1: Load from DAMIT CSV format
def load_damit_csv(csv_path):
    """Load DAMIT CSV lightcurve data."""
    df = pd.read_csv(csv_path, comment='#')
    df = df.dropna()
    data = df.values
    return [data]  # Return as single epoch

# Example: Load asteroid 101
# epochs = load_damit_csv("data/damit_csv_qf_ge_3/asteroid_101.csv")

# For this demo, we'll create synthetic data
print("Creating synthetic lightcurve data for demonstration...")

# Generate synthetic observations
n_obs = 100
t = np.linspace(0, 10, n_obs)  # 10 days
period_hours = 8.5
brightness = 1.0 + 0.2 * np.sin(2 * np.pi * t * 24 / period_hours)

# Geometry (sun and observer positions)
sun_x = np.full(n_obs, 0.5)
sun_y = np.full(n_obs, -0.7)
sun_z = np.full(n_obs, 0.5)
obs_x = np.full(n_obs, 0.3)
obs_y = np.full(n_obs, -0.6)
obs_z = np.full(n_obs, 0.7)

# Combine into epoch array
epoch = np.column_stack([t + 2450000, brightness, sun_x, sun_y, sun_z, obs_x, obs_y, obs_z])
epochs = [epoch]

print(f"✓ Loaded {len(epochs)} epoch(s) with {sum(len(e) for e in epochs)} total observations")

## Step 3: Run Analysis

Use the `analyze()` function to estimate period and predict poles.

In [ ]:
# Run analysis
print("Running analysis...")

result = analyze(
    epochs=epochs,
    object_id="demo_asteroid",
    fold=0
)

print("Analysis complete!")

## Step 4: View Results

Display the analysis results in a user-friendly format.

In [ ]:
# Print results
print("=" * 80)
print(f"POLE PREDICTION RESULTS: {result.object_id}")
print("=" * 80)
print()

# Period
period = result.period
print(f"Period: {period.period_hours:.2f} ± {period.uncertainty_hours:.2f} h")
print(f"95% CI: [{period.ci_low_hours:.2f}, {period.ci_high_hours:.2f}] h")
print()

# Best pole
best = result.best_pole
print("Best Pole:")
print(f"  λ = {best.lambda_deg:.1f}° (ecliptic longitude)")
print(f"  β = {best.beta_deg:.1f}° (ecliptic latitude)")
print(f"  Quality score: {best.score:.3f}")
print(f"  Period alias: {best.alias}")
print()

# All candidates
print(f"All {len(result.poles)} candidates (3 periods × 3 poles):")
print(f"{'#':<3} {'Alias':<8} {'Period':<9} {'λ':<8} {'β':<8} {'Score':<6}")
print("-" * 50)

for i, pole in enumerate(result.poles, 1):
    print(
        f"{i:<3} {pole.alias:<8} {pole.period_hours:>7.2f}h "
        f"{pole.lambda_deg:>6.1f}° {pole.beta_deg:>6.1f}° {pole.score:>6.3f}"
    )

print()

# Uncertainty
unc = result.uncertainty
print("Uncertainty:")
print(f"  Spread: {unc.spread_deg:.1f}°")
print(f"  Confidence: {unc.confidence:.2f}")
print("=" * 80)

## Step 5: Understanding the Results

### What do these numbers mean?

**Period**:
- The estimated rotation period in hours
- Uncertainty is 1-sigma (68% confidence)
- 95% CI is the credible interval

**Poles**:
- lambda: Ecliptic longitude [0 deg, 360 deg)
- beta: Ecliptic latitude [-90 deg, +90 deg]
- Score: All candidates have equal scores (no quality head in production model)

**Aliases**:
- `base`: Estimated period P
- `double`: 2x period (2P)
- `half`: 0.5x period (P/2), only if P >= 8h

### Why 6-9 candidates?

The model produces **3 pole candidates** for each period alias:
- Period aliases: base, double, half (2 or 3)
- Poles per alias: K=3
- Total: 6-9 candidates

### How good are the predictions?

- **Oracle@K=3 mean**: 19.02 deg (if you know which of K=3 is best)
- **Pooled median**: 16.61 deg
- **ZTF external validation**: 18.82 deg on 163 asteroids

The candidates are **unranked**. Users should evaluate all candidates using external validation (radar, occultations, other lightcurve inversions, etc.).

---

## Step 6: Visualization (Optional)

Visualize the pole candidates on the celestial sphere.

In [ ]:
# Visualization requires matplotlib
try:
    import matplotlib.pyplot as plt
    from matplotlib.patches import Circle
    
    # Create figure
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Plot 1: All pole candidates on ecliptic projection
    for i, pole in enumerate(result.poles, 1):
        color = 'red' if i == 1 else 'blue'
        size = 100 if i == 1 else 30
        label = 'Best' if i == 1 else None
        ax1.scatter(pole.lambda_deg, pole.beta_deg, 
                   s=size, c=color, alpha=0.7, label=label, edgecolors='black')
        ax1.text(pole.lambda_deg + 5, pole.beta_deg + 5, str(i), fontsize=8)
    
    ax1.set_xlabel('Ecliptic Longitude λ (°)', fontsize=12)
    ax1.set_ylabel('Ecliptic Latitude β (°)', fontsize=12)
    ax1.set_xlim(0, 360)
    ax1.set_ylim(-90, 90)
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    ax1.set_title('Pole Candidates (Ecliptic Coordinates)', fontsize=14, fontweight='bold')
    
    # Plot 2: Quality scores
    scores = [pole.score for pole in result.poles]
    colors_bar = ['red' if i == 0 else 'blue' for i in range(len(scores))]
    ax2.bar(range(1, len(scores) + 1), scores, color=colors_bar, alpha=0.7, edgecolor='black')
    ax2.set_xlabel('Candidate #', fontsize=12)
    ax2.set_ylabel('Quality Score', fontsize=12)
    ax2.set_ylim(0, 1)
    ax2.grid(True, alpha=0.3, axis='y')
    ax2.set_title('Quality Scores', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("matplotlib not available - skipping visualization")

## Step 7: Save Results (Optional)

Save results to JSON for further analysis.

In [ ]:
# Save to JSON
def save_results_json(result, output_path):
    """Save analysis results to JSON file."""
    
    def to_python(val):
        """Convert numpy types to Python types."""
        if hasattr(val, 'item'):
            return val.item()
        return val
    
    data = {
        "object_id": result.object_id,
        "period": {
            "period_hours": to_python(result.period.period_hours),
            "uncertainty_hours": to_python(result.period.uncertainty_hours),
            "ci_low_hours": to_python(result.period.ci_low_hours),
            "ci_high_hours": to_python(result.period.ci_high_hours),
        },
        "best_pole": {
            "lambda_deg": to_python(result.best_pole.lambda_deg),
            "beta_deg": to_python(result.best_pole.beta_deg),
            "xyz": [to_python(x) for x in result.best_pole.xyz],
            "score": to_python(result.best_pole.score),
        },
        "all_poles": [
            {
                "lambda_deg": to_python(pole.lambda_deg),
                "beta_deg": to_python(pole.beta_deg),
                "xyz": [to_python(x) for x in pole.xyz],
                "period_hours": to_python(pole.period_hours),
                "alias": pole.alias,
                "score": to_python(pole.score),
            }
            for pole in result.poles
        ],
        "uncertainty": {
            "spread_deg": to_python(result.uncertainty.spread_deg),
            "confidence": to_python(result.uncertainty.confidence),
        }
    }
    
    with open(output_path, 'w') as f:
        json.dump(data, f, indent=2)
    
    print(f"✓ Results saved to: {output_path}")

# Example: Save results
# save_results_json(result, "demo_result.json")

## Step 8: Batch Processing (Optional)

Process multiple asteroids at once.

In [ ]:
def batch_process(input_dir, fold=0):
    """Process all CSV files in a directory."""
    from pathlib import Path
    
    input_path = Path(input_dir)
    csv_files = list(input_path.glob("*.csv"))
    
    print(f"Found {len(csv_files)} CSV file(s) in {input_dir}")
    print()
    
    results = []
    
    for i, csv_file in enumerate(csv_files, 1):
        print(f"[{i}/{len(csv_files)}] Processing: {csv_file.name}")
        
        try:
            # Load data
            epochs = load_damit_csv(csv_file)
            object_id = csv_file.stem
            
            # Analyze
            result = analyze(epochs=epochs, object_id=object_id, fold=fold)
            results.append(result)
            
            print(f"  ✓ Period={result.period.period_hours:.2f}h, "
                  f"Pole=({result.best_pole.lambda_deg:.1f}°, {result.best_pole.beta_deg:.1f}°)")
        
        except Exception as e:
            print(f"  ✗ Failed: {e}")
        
        print()
    
    return results

# Example: Batch process all asteroids
# results = batch_process("data/damit_csv_qf_ge_3", fold=0)

## Troubleshooting

### Common Issues

**1. Import Error: "No module named 'lc_pipeline'"**
- Solution: Install lc_pipeline with `!pip install -e /path/to/lc_pipeline`

**2. Period estimation fails**
- Cause: Insufficient data or very noisy lightcurve
- Solution: Check that you have at least 50-100 observations

**3. All poles have same score**
- Cause: Quality head not confident
- Solution: This is expected for some asteroids - the model is uncertain

**4. CUDA out of memory**
- Solution: The model will automatically fall back to CPU if CUDA fails

### Getting Help

- Documentation: See `docs/` directory
- Issues: Report bugs on GitHub
- Examples: See `example.py` and `docs/EXAMPLES.md`

---

## Summary

You've successfully:
1. ✓ Installed lc_pipeline
2. ✓ Loaded lightcurve data
3. ✓ Estimated period
4. ✓ Predicted poles
5. ✓ Visualized results

**Next Steps**:
- Try with your own asteroid data
- Experiment with different folds (0-4)
- Compare results across multiple periods
- Validate with external data sources

**Remember**: The model is not perfect. Always validate critical results with other methods!
